[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/feat/refine/examples/benchmark.ipynb)


# MISDA - Maximal Independent Structural Dimensionality Analysis

This notebook serves as a comprehensive benchmark suite for the Maximal Independent Structural Dimensionality Analysis (MISDA) framework.
It systematically evaluates the algorithm's efficacy across a spectrum of synthetic benchmarks, ranging from canonical correlation patterns—including linear redundancies and latent manifolds—to complex Multi-Objective Problems (MOPs). The analysis verifies MISDA's capability to correctly identify intrinsic dimensionality and preserve the topological fidelity of the Pareto frontier.

In [ ]:
# Install MISDA 
!pip install --upgrade git+https://github.com/monacofj/misda.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import misda

# Reload for development iteration
import importlib
importlib.reload(misda)

print("MISDA imported successfully.")


## 1. Data Generators & Utilities

### General Helpers

In [ ]:
from benchmarks.cases import (
    make_case1_independence,
    make_case2_total_redundancy,
    make_case3_block_structure,
    make_case4_two_big_blocks,
    make_case5_chain_structure,
    make_case6_mixed_structure,
    make_case7_pure_conflict_groups,
)
from mop_definitions import (
    mopA_monotonic_redundancy,
    mopB_tradeoff_with_redundancies,
    mopC_latent_blocks_4x5,
    mopD_pure_conflict_groups,
    mopE_partial_redundancy_noisy,
    mopF_regime_switching,
)


### Validation Utilities
Custom function to evaluate reconstruction fidelity on benchmark datasets.

### Run cases

In [ ]:
def run_cases(cases_list, N=300):
    results = {}
    for name, gen in cases_list:
        Y, truth = gen(N=N)
        problem = truth.get("feature", truth.get("notes", ""))
        intuition = truth.get("intuition", "")
        graph_expected = truth.get("graph_expected", "")
        latent = truth.get("latent_expected", truth.get("intrinsic_dim_expected", ""))
        structural = truth.get("structural_expected", truth.get("latent_expected", truth.get("intrinsic_dim_expected", "")))

        print(f"\n{'='*80}")
        print("Case description")
        print(f"Running:    {name}")
        if problem:
            print(f"Problem:    {problem}")
        if intuition:
            print(f"Intuition:  {intuition}")
        if latent != "" and latent is not None:
            print(f"Latent:     {latent}")
        if structural != "" and structural is not None:
            print(f"Structural: {structural}")
        if graph_expected:
            print(f"Graph:      {graph_expected}")
        print(f"{'='*80}")

        # --- 1. Execute MISDA analysis (using defaults: )
        result = misda.analyze(Y, name=name)
        result.validate()
        
        # --- 2. Full Technical Report ---
        print(result.report())

        # --- 3. Plot Graph ---
        try:
            result.plot()
        except Exception as e:
            print(f"Plotting failed: {e}")
        
        results[name] = {
            "result_obj": result,
            "truth": truth
        }
    return results


### Canonical Structure Test Suite (qualitative calibration)

In [ ]:
# Canonical generators are imported from benchmarks.cases.


### Synthetic MOP Test Suite (nferential validation)

In [ ]:
# MOP generators are imported from mop_definitions.


## Run test batteries

In [ ]:
battery1 = [
    ("Case 1 - Total independence", make_case1_independence),
    ("Case 2 - Total redundancy", make_case2_total_redundancy),
    ("Case 3 - Blocks (4 x 5)", make_case3_block_structure),
    ("Case 4 - Blocks (2 x 10)", make_case4_two_big_blocks),
    ("Case 5 - Chain", make_case5_chain_structure),
    ("Case 6 - Mixed (indep + latents)", make_case6_mixed_structure),
    ("Case 7 - Structural conflict (anti-corr) with groups", make_case7_pure_conflict_groups),
]

print("\n=== RUNNING STANDARD CORRELATION BATTERY ===")
battery1_results = run_cases(battery1)

battery1_fidelity_df = misda.compile_benchmark_summary(battery1_results)
print("\n--- MISDA Reconstruction Fidelity Evaluation for Canonical Cases ---")
print(battery1_fidelity_df.to_markdown(index=False))


In [ ]:
battery2 = [
    ("MOP-A — Monotonic redundancy", mopA_monotonic_redundancy),
    ("MOP-B — Trade-off + redundancies", mopB_tradeoff_with_redundancies),
    ("MOP-C — Latent blocks", mopC_latent_blocks_4x5),
    ("MOP-D — Pure conflict groups", mopD_pure_conflict_groups),
    ("MOP-E — Partial redundancy + noise", mopE_partial_redundancy_noisy),
    ("MOP-F — Regime switching", mopF_regime_switching),
]

print("\n\n=== RUNNING MOP BENCHMARK BATTERY ===")
mop_results = run_cases(battery2)

mop_fidelity_df = misda.compile_benchmark_summary(mop_results)
print("\n--- MISDA Reconstruction Fidelity Evaluation for MOP Cases ---")
print(mop_fidelity_df.to_markdown(index=False))


# 3. Conclusions

The benchmarking suite evaluates the behavior of the MISDA algorithm across a diverse set of structural topologies, yielding the following observations:

### Resolving the Signal from the Noise
In the canonical test cases, the algorithm distinguishes between varying types of dependence. It removed full linear redundancies (Case 2) while maintaining the dimensions of independent variables (Case 1). In more complex scenarios, such as the "Chain" structure (Case 5) or "Mixed Structure" (Case 6), the algorithm identified the underlying latent drivers by analyzing the connectivity of the partial correlation graph, rather than solely pairwise relationships. Limitations were noted in Case 4 (Transitive), where the algorithm relies on the chosen alpha threshold to distinguish indirect from direct correlations.

### The Pragmatism of MISDA in MOPs
When applied to synthetic Multi-Objective Problems (MOPs), which mimic nonlinear engineering landscapes, the results highlighted specific behaviors:
*   **MOP-A (Monotonic Redundancy):** The algorithm reduced the 20 transformed objectives to the single underlying variable, confirming robustness to monotonic transformations.
*   **MOP-D (Conflict Groups):** By identifying independent sets, the algorithm preserved the representative objectives from conflicting groups, retaining the dimensional structure of the trade-offs.
*   **MOP-B & C (Non-Linear):** The method showed sensitivity to highly non-linear relationships, where the linear correlation assumption ($r$) may underestimate dependence.
*   **MOP-E & F (Regime Switching):** In cases of low signal-to-noise ratio, the "Caution" parameter and the adaptive alpha threshold determined whether the algorithm defaulted to a conservative (keep all) or aggressive (prune) strategy.

Ultimately, these benchmarks indicate that MISDA functions as a **structural inference engine**. It separates essential conflict from redundant noise, providing a minimal representation of the decision space that attempts to preserve the integrity of the Pareto frontier.